# Superstore Sales EDA: 03. Exploratory Data Analysis
---
**Objective:** Deep statistical exploration across univariate distributions, category economics, promotional discount sensitivity, regional profit polarizations, fulfillment lead times, and time series seasonality.


In [1]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

def get_path(rel_path):
    for candidate in [Path(rel_path), Path("..") / rel_path, Path("/home/adarsh/Documents/Superstore Sales EDA") / rel_path]:
        if candidate.exists():
            return candidate
    return Path(rel_path)

data_path = get_path("data/cleaned/superstore_featured.csv")
df = pd.read_csv(data_path)
df["Order Date"] = pd.to_datetime(df["Order Date"])
print(f"Loaded featured dataset: {df.shape[0]:,} rows, {df.shape[1]} columns.")


Loaded featured dataset: 9,994 rows, 34 columns.


## 1. High-Level Portfolio Performance & Univariate Summary
Inspect top-level commercial KPIs: Total Sales, Total Profit, Aggregate Margin %, and Loss Order Share.


In [2]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
overall_margin = (total_profit / total_sales) * 100
loss_orders = df["Is_Loss"].sum()
loss_rate = (loss_orders / len(df)) * 100

print(f"Total Revenue:       ${total_sales:,.2f}")
print(f"Total Net Profit:    ${total_profit:,.2f}")
print(f"Overall Margin:      {overall_margin:.2f}%")
print(f"Loss-Making Orders:  {loss_orders:,} of {len(df):,} ({loss_rate:.2f}%)")


Total Revenue:       $2,297,200.86
Total Net Profit:    $286,397.02
Overall Margin:      12.47%
Loss-Making Orders:  1,871 of 9,994 (18.72%)


## 2. Category & Sub-Category Performance
Analyze gross sales vs net profit across Categories and identify chronic loss-generating sub-categories.


In [3]:
cat_df = df.groupby("Category").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Order_Count=("Row ID", "count")
).reset_index()
cat_df["Margin%"] = (cat_df["Profit"] / cat_df["Sales"]) * 100
print(cat_df.sort_values(by="Profit", ascending=False))


          Category        Sales       Profit  Order_Count    Margin%
2       Technology  836154.0330  145454.9481         1847  17.395712
1  Office Supplies  719047.0320  122490.8008         6026  17.035158
0        Furniture  741999.7953   18451.2728         2121   2.486695


In [4]:
sub_df = df.groupby(["Category", "Sub-Category"]).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Row ID", "count"),
    Avg_Discount=("Discount", "mean")
).reset_index()
sub_df["Margin%"] = (sub_df["Profit"] / sub_df["Sales"]) * 100
sub_df = sub_df.sort_values(by="Profit", ascending=False)

print("Top 5 Profit Generating Sub-Categories:")
print(sub_df.head(5))

print("\nBottom 5 Sub-Categories (Loss Makers):")
print(sub_df.tail(5))


Top 5 Profit Generating Sub-Categories:
           Category Sub-Category       Sales      Profit  Orders  \
14       Technology      Copiers  149528.030  55617.8249      68   
16       Technology       Phones  330007.054  44515.7306     889   
13       Technology  Accessories  167380.318  41936.6357     775   
10  Office Supplies        Paper   78479.206  34053.5693    1370   
6   Office Supplies      Binders  203412.733  30221.7633    1523   

    Avg_Discount    Margin%  
14      0.161765  37.195585  
16      0.154556  13.489327  
13      0.078452  25.054700  
10      0.074891  43.391837  
6       0.372292  14.857361  

Bottom 5 Sub-Categories (Loss Makers):
           Category Sub-Category        Sales      Profit  Orders  \
15       Technology     Machines  189238.6310   3384.7569     115   
8   Office Supplies    Fasteners    3024.2800    949.5182     217   
12  Office Supplies     Supplies   46673.5380  -1189.0995     190   
0         Furniture    Bookcases  114879.9963  -3472.55

## 3. Promotional Discount Analysis: The 20% Profit Cliff
Investigate how discount rates directly correlate with margin erosion and order unprofitability.


In [5]:
disc_perf = df.groupby("Discount_Bucket", observed=False).agg(
    Orders=("Row ID", "count"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Loss_Orders=("Is_Loss", "sum")
).reset_index()

disc_perf["Margin%"] = (disc_perf["Total_Profit"] / disc_perf["Total_Sales"]) * 100
disc_perf["Loss_Rate%"] = (disc_perf["Loss_Orders"] / disc_perf["Orders"]) * 100
print(disc_perf)


   Discount_Bucket  Orders   Total_Sales  Total_Profit  Loss_Orders  \
0        0% (None)    4798  1.087908e+06   320987.6032            0   
1      1-20% (Low)    3803  8.465222e+05   100785.4745          523   
2  21-50% (Medium)     537  2.985414e+05   -58817.0047          492   
3      50%+ (High)     856  6.422874e+04   -76559.0513          856   

      Margin%  Loss_Rate%  
0   29.505019    0.000000  
1   11.905827   13.752301  
2  -19.701456   91.620112  
3 -119.197502  100.000000  


## 4. Geographic & State-Level Polarization
Identify top-earning states (California, New York) and chronic loss states (Texas, Ohio, Pennsylvania, Illinois).


In [6]:
state_perf = df.groupby("State").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Row ID", "count"),
    Avg_Discount=("Discount", "mean")
).reset_index()
state_perf["Margin%"] = (state_perf["Profit"] / state_perf["Sales"]) * 100

print("Top 5 Profitable States:")
print(state_perf.sort_values(by="Profit", ascending=False).head(5))

print("\nBottom 5 Loss States:")
print(state_perf.sort_values(by="Profit", ascending=True).head(5))


Top 5 Profitable States:
         State        Sales      Profit  Orders  Avg_Discount    Margin%
3   California  457687.6315  76381.3871    2001      0.072764  16.688541
30    New York  310876.2710  74038.5486    1128      0.055319  23.816082
45  Washington  138641.2700  33402.6517     506      0.064032  24.092863
20    Michigan   76269.6140  24463.1876     255      0.007059  32.074618
44    Virginia   70636.7200  18597.9504     224      0.000000  26.329012

Bottom 5 Loss States:
             State        Sales      Profit  Orders  Avg_Discount    Margin%
41           Texas  170188.0458 -25729.3563     985      0.370193 -15.118192
33            Ohio   78258.1360 -16971.3766     469      0.324947 -21.686405
36    Pennsylvania  116511.9140 -15559.9603     587      0.328620 -13.354823
11        Illinois   80166.1010 -12607.8870     492      0.390041 -15.727205
31  North Carolina   55603.1640  -7490.9122     249      0.283534 -13.472097


## 5. Shipping Modes & Logistics Lead Times
Examine fulfillment lead times and profitability across fulfillment tiers.


In [7]:
ship_perf = df.groupby("Ship Mode").agg(
    Mean_Days=("Shipping_Days", "mean"),
    Median_Days=("Shipping_Days", "median"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Row ID", "count")
).reset_index()
ship_perf["Margin%"] = (ship_perf["Total_Profit"] / ship_perf["Total_Sales"]) * 100
print(ship_perf)


        Ship Mode  Mean_Days  Median_Days   Total_Sales  Total_Profit  Orders  \
0     First Class   2.182705          2.0  3.514284e+05    48969.8399    1538   
1        Same Day   0.044199          0.0  1.283631e+05    15891.7589     543   
2    Second Class   3.238046          3.0  4.591936e+05    57446.6354    1945   
3  Standard Class   5.006535          5.0  1.358216e+06   164088.7875    5968   

     Margin%  
0  13.934513  
1  12.380315  
2  12.510331  
3  12.081202  


## 6. Time Series & Seasonality Analysis
Resample monthly sales and profit to visualize annual Q4 surges and growth trajectories.


In [8]:
monthly_trend = df.set_index("Order Date").resample("ME").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum")
).reset_index()
monthly_trend["Margin%"] = (monthly_trend["Profit"] / monthly_trend["Sales"]) * 100

print("Annual Summary:")
annual = df.groupby("Order_Year").agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
annual["Margin%"] = (annual["Profit"] / annual["Sales"]) * 100
annual["YoY_Sales_Growth%"] = annual["Sales"].pct_change() * 100
print(annual)


Annual Summary:
                  Sales      Profit    Margin%  YoY_Sales_Growth%
Order_Year                                                       
2014        484247.4981  49543.9741  10.231126                NaN
2015        470532.5090  61618.6037  13.095504          -2.832227
2016        609205.5980  81795.1743  13.426530          29.471521
2017        733215.2552  93439.2696  12.743771          20.355962
